# Introduction


We analyze creatively the data in **NYC Taxi Trip Duration** dataset.  

The city is viewed as a living organism, that pulse, flow, and breathes.

Note: the visualizations are the result of random sampling 200K rows from the entire dataset, with 1.45M rows.

# Preparations

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans

import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
df = pd.read_csv("/kaggle/input/datasets/yasserh/nyc-taxi-trip-duration/NYC.csv")

# Data glimpse

In [3]:
df.head()

,id,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,2016-03-14 17:32:30,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,2016-06-12 00:54:38,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,2016-01-19 12:10:48,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,2016-04-06 19:39:40,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,2016-03-26 13:38:10,1,-73.973053,40.793209,-73.972923,40.782520,N,435


# Basic cleaning

In [4]:
TIME_COL = "pickup_datetime"
PICKUP_LAT = "pickup_latitude"
PICKUP_LON = "pickup_longitude"
DROPOFF_LAT = "dropoff_latitude"
DROPOFF_LON = "dropoff_longitude"

# Keep only needed columns
cols = [TIME_COL, PICKUP_LAT, PICKUP_LON, DROPOFF_LAT, DROPOFF_LON]
df = df[cols].copy()

# Drop missing values
df = df.dropna()

# Convert datetime
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.dropna(subset=[TIME_COL])

# Rough geographic filtering
# Example bounds for NYC; adjust for your city
df = df[
    df[PICKUP_LAT].between(40.5, 41.0) &
    df[PICKUP_LON].between(-74.3, -73.6) &
    df[DROPOFF_LAT].between(40.5, 41.0) &
    df[DROPOFF_LON].between(-74.3, -73.6)
].copy()

# Sample for performance
if len(df) > 200_000:
    df = df.sample(200_000, random_state=42)

df.shape

(200000, 5)

# Feature engineering

In [5]:
df["hour"] = df[TIME_COL].dt.hour
df["day_of_week"] = df[TIME_COL].dt.day_name()
df["is_weekend"] = df[TIME_COL].dt.weekday >= 5
df["date"] = df[TIME_COL].dt.date

# Get the pulse of the city

In [6]:
hourly = (
    df.groupby("hour")
      .size()
      .reset_index(name="trip_count")
      .sort_values("hour")
)

fig = px.line(
    hourly,
    x="hour",
    y="trip_count",
    markers=True,
    title="The Pulse of the City: Trips by Hour"
)
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Number of Trips",
    template="plotly_dark"
)
fig.show()

# Weekday vs weekend pulse

In [7]:
hourly_type = (
    df.groupby(["hour", "is_weekend"])
      .size()
      .reset_index(name="trip_count")
)

hourly_type["day_type"] = np.where(hourly_type["is_weekend"], "Weekend", "Weekday")

fig = px.line(
    hourly_type,
    x="hour",
    y="trip_count",
    color="day_type",
    markers=True,
    title="Weekday vs Weekend Mobility Rhythm"
)
fig.update_layout(
    xaxis_title="Hour of Day",
    yaxis_title="Trips",
    template="plotly_dark"
)
fig.show()

# Breathing city

Let's look to an animated pickup density over the day & night.

In [8]:
# Create hourly aggregated pickup points
density_df = df[[PICKUP_LAT, PICKUP_LON, "hour"]].copy()
density_df = density_df.rename(columns={
    PICKUP_LAT: "lat",
    PICKUP_LON: "lon"
})

fig = px.density_map(
    density_df,
    lat="lat",
    lon="lon",
    z=None,
    radius=8,
    animation_frame="hour",
    center={"lat": density_df["lat"].mean(), "lon": density_df["lon"].mean()},
    zoom=10,
    map_style="carto-darkmatter",
    title="The City Breathes: Pickup Density by Hour"
)

fig.update_layout(margin=dict(l=0, r=0, t=50, b=0))
fig.show()

# The city's center of life

In [9]:
center_of_life = (
    df.groupby("hour")
      .agg(
          center_lat=(PICKUP_LAT, "mean"),
          center_lon=(PICKUP_LON, "mean"),
          trip_count=(PICKUP_LAT, "size")
      )
      .reset_index()
)

fig = px.scatter_map(
    center_of_life,
    lat="center_lat",
    lon="center_lon",
    size="trip_count",
    hover_name="hour",
    center={"lat": center_of_life["center_lat"].mean(), "lon": center_of_life["center_lon"].mean()},
    zoom=10,
    map_style="carto-darkmatter",
    title="The Migrating Heartbeat of the City"
)

fig.add_trace(
    go.Scattermap(
        lat=center_of_life["center_lat"],
        lon=center_of_life["center_lon"],
        mode="lines+markers",
        text=center_of_life["hour"],
        name="Daily movement",
    )
)

fig.show()

# Identify mobility zones

We will use clustering to identify the mobility zones.

In [10]:
pickup_coords = df[[PICKUP_LAT, PICKUP_LON]].copy()
pickup_coords.columns = ["lat", "lon"]

n_zones = 20
kmeans = KMeans(n_clusters=n_zones, random_state=42, n_init=10)
df["pickup_zone"] = kmeans.fit_predict(pickup_coords)

dropoff_coords = df[[DROPOFF_LAT, DROPOFF_LON]].copy()
dropoff_coords.columns = ["lat", "lon"]

df["dropoff_zone"] = kmeans.predict(dropoff_coords)

# Visualize flows between zones

After we build the city zones, let's see how these interact.

In [11]:
flows = (
    df.groupby(["pickup_zone", "dropoff_zone"])
      .size()
      .reset_index(name="trip_count")
)

flows = flows[flows["pickup_zone"] != flows["dropoff_zone"]]
flows = flows.sort_values("trip_count", ascending=False).head(100)

zone_centers = pd.DataFrame(kmeans.cluster_centers_, columns=["lat", "lon"])
zone_centers["zone"] = zone_centers.index

flows = flows.merge(
    zone_centers.rename(columns={"zone": "pickup_zone", "lat": "pickup_lat", "lon": "pickup_lon"}),
    on="pickup_zone",
    how="left"
)

flows = flows.merge(
    zone_centers.rename(columns={"zone": "dropoff_zone", "lat": "dropoff_lat", "lon": "dropoff_lon"}),
    on="dropoff_zone",
    how="left"
)

flows.head()

,pickup_zone,dropoff_zone,trip_count,pickup_lat,pickup_lon,dropoff_lat,dropoff_lon
0,16,0,3082,40.753068,-73.992043,40.763772,-73.983922
1,2,13,2981,40.759934,-73.969930,40.770126,-73.959366
2,13,2,2699,40.770126,-73.959366,40.759934,-73.969930
3,13,10,2693,40.770126,-73.959366,40.780822,-73.952156
4,0,16,2626,40.763772,-73.983922,40.753068,-73.992043


# Drawing the arterial flows

In [12]:
fig = go.Figure()

for _, row in flows.iterrows():
    fig.add_trace(go.Scattermapbox(
        mode="lines",
        lon=[row["pickup_lon"], row["dropoff_lon"]],
        lat=[row["pickup_lat"], row["dropoff_lat"]],
        line=dict(width=max(1, row["trip_count"] / flows["trip_count"].max() * 8)),
        hovertext=f'Zone {row["pickup_zone"]} → Zone {row["dropoff_zone"]}<br>Trips: {row["trip_count"]}',
        hoverinfo="text",
        showlegend=False
    ))

fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_zoom=10,
    mapbox_center={
        "lat": df[PICKUP_LAT].mean(),
        "lon": df[PICKUP_LON].mean()
    },
    title="Arteries and Veins: Strongest Mobility Flows",
    template="plotly_dark",
    margin=dict(l=0, r=0, t=40, b=0)
)

fig.show()

# Compare circulation during day

In [13]:
def build_flows_for_hours(data, hours, top_n=40):
    temp = data[data["hour"].isin(hours)]
    temp_flows = (
        temp.groupby(["pickup_zone", "dropoff_zone"])
            .size()
            .reset_index(name="trip_count")
    )
    temp_flows = temp_flows[temp_flows["pickup_zone"] != temp_flows["dropoff_zone"]]
    temp_flows = temp_flows.sort_values("trip_count", ascending=False).head(top_n)

    temp_flows = temp_flows.merge(
        zone_centers.rename(columns={"zone": "pickup_zone", "lat": "pickup_lat", "lon": "pickup_lon"}),
        on="pickup_zone", how="left"
    )
    temp_flows = temp_flows.merge(
        zone_centers.rename(columns={"zone": "dropoff_zone", "lat": "dropoff_lat", "lon": "dropoff_lon"}),
        on="dropoff_zone", how="left"
    )
    return temp_flows

morning_flows = build_flows_for_hours(df, [6, 7, 8, 9], top_n=40)
evening_flows = build_flows_for_hours(df, [17, 18, 19, 20], top_n=40)

## Morning map

In [14]:
fig = go.Figure()

for _, row in morning_flows.iterrows():
    fig.add_trace(go.Scattermapbox(
        mode="lines",
        lon=[row["pickup_lon"], row["dropoff_lon"]],
        lat=[row["pickup_lat"], row["dropoff_lat"]],
        line=dict(width=max(1, row["trip_count"] / morning_flows["trip_count"].max() * 8)),
        hoverinfo="text",
        hovertext=f'Morning: {row["pickup_zone"]} → {row["dropoff_zone"]} ({row["trip_count"]} trips)',
        showlegend=False
    ))

fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_zoom=10,
    mapbox_center={"lat": df[PICKUP_LAT].mean(), "lon": df[PICKUP_LON].mean()},
    title="Morning Circulation",
    template="plotly_dark",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

## Evening map

In [15]:
fig = go.Figure()

for _, row in evening_flows.iterrows():
    fig.add_trace(go.Scattermapbox(
        mode="lines",
        lon=[row["pickup_lon"], row["dropoff_lon"]],
        lat=[row["pickup_lat"], row["dropoff_lat"]],
        line=dict(width=max(1, row["trip_count"] / evening_flows["trip_count"].max() * 8)),
        hoverinfo="text",
        hovertext=f'Evening: {row["pickup_zone"]} → {row["dropoff_zone"]} ({row["trip_count"]} trips)',
        showlegend=False
    ))

fig.update_layout(
    mapbox_style="carto-darkmatter",
    mapbox_zoom=10,
    mapbox_center={"lat": df[PICKUP_LAT].mean(), "lon": df[PICKUP_LON].mean()},
    title="Evening Circulation",
    template="plotly_dark",
    margin=dict(l=0, r=0, t=40, b=0)
)
fig.show()

## Circadian phases

In [16]:
def phase_from_hour(h):
    if 1 <= h <= 5:
        return "Sleep", "01–05"
    elif 6 <= h <= 9:
        return "Awakening", "06–09"
    elif 10 <= h <= 16:
        return "Active Metabolism", "10–16"
    elif 17 <= h <= 20:
        return "Stress Response", "17–20"
    else:
        return "Recovery / Nightlife", "21-24 & 00"

df[["phase", "interval"]] = df["hour"].apply(
    lambda h: pd.Series(phase_from_hour(h))
)

phase_order = [
    "Sleep",
    "Awakening",
    "Active Metabolism",
    "Stress Response",
    "Recovery / Nightlife"
]

phase_counts = (
    df.groupby(["phase", "interval"])
      .size()
      .reset_index(name="trip_count")
)

# enforce order
phase_counts["phase"] = pd.Categorical(
    phase_counts["phase"],
    categories=phase_order,
    ordered=True
)

phase_counts = phase_counts.sort_values("phase")

fig = px.bar(
    phase_counts,
    x="phase",
    y="trip_count",
    # intervals go on bars
    text="interval",   
    title="Circadian Rhythm of the City"
)

fig.update_traces(
    # puts intervals above bars
    textposition="outside"  
)

fig.update_layout(
    template="plotly_dark",
    xaxis_title="Biological Phase",
    yaxis_title="Number of Trips"
)

fig.show()

# Looking for anomalies

In [17]:
daily_counts = (
    df.groupby("date")
      .size()
      .reset_index(name="trip_count")
)

daily_counts["rolling_mean"] = daily_counts["trip_count"].rolling(7, min_periods=1).mean()
daily_counts["rolling_std"] = daily_counts["trip_count"].rolling(7, min_periods=1).std().fillna(0)
daily_counts["z_score"] = (
    (daily_counts["trip_count"] - daily_counts["rolling_mean"]) /
    daily_counts["rolling_std"].replace(0, np.nan)
)

fig = px.line(
    daily_counts,
    x="date",
    y="trip_count",
    title="Daily Mobility Volume"
)
fig.update_layout(template="plotly_dark")
fig.show()

anomalies = daily_counts[daily_counts["z_score"].abs() > 2]
anomalies

,date,trip_count,rolling_mean,rolling_std,z_score
17,2016-01-18,937,1096.285714,77.409548,-2.057701
22,2016-01-23,222,978.000000,343.662625,-2.199832


## Final Thoughts

Urban mobility does not just describe where people go. It reveals that the city behaves like a living system: it pulses with activity, circulates through major corridors, shifts its center of energy over time, and follows a daily rhythm of activation and recovery.

By framing trips as pulse, breath, and flow, we move from a transport map to a biological portrait of the city.

# 